In [1]:
import pandas as pd
import numpy as np

CSV_PATH = "oklahoma/NEW_NBE_and_NBI_OK.csv"
TARGET_COL = "HI"
YEAR_COL = "YEAR"
BRIDGE_ID_COL = "STRUCTURE_NUMBER"
ELEM_NUM_COL = "ELEMENT_NUMBER"
PARENT_ELEM_NUM_COL = "PARENT_ELEMENT_NUMBER"

YEARS = [2020, 2021, 2022, 2023, 2024]

def normalize_bridge_id(x) -> str:
    """
    STRUCTURE_NUMBER loads as float sometimes (e.g., 5550000000000.0).
    Convert to a stable string ID without losing precision.
    """
    if pd.isna(x):
        return None
    # Use integer-string if possible
    try:
        return str(int(float(x)))
    except Exception:
        return str(x)

def build_global_node_index(df: pd.DataFrame):
    """
    Creates a global node index across all years:
      - bridge:{bridge_id}
      - elem:{bridge_id}:{element_number}
    Returns:
      node2idx: dict[str, int]
      idx2node: list[str]
      elem_nodes: set[str]  (node keys for elements)
      bridge_nodes: set[str]
    """
    bridge_ids = df[BRIDGE_ID_COL].map(normalize_bridge_id)
    elem_nums = df[ELEM_NUM_COL].astype("Int64")

    bridge_nodes = set(f"bridge:{bid}" for bid in bridge_ids.dropna().unique())
    elem_nodes = set(
        f"elem:{bid}:{int(en)}"
        for bid, en in zip(bridge_ids, elem_nums)
        if (bid is not None) and (not pd.isna(en))
    )

    all_nodes = sorted(list(bridge_nodes | elem_nodes))
    node2idx = {k: i for i, k in enumerate(all_nodes)}
    return node2idx, all_nodes, elem_nodes, bridge_nodes

def build_static_edges(df: pd.DataFrame, node2idx: dict):
    """
    Builds a static edge_index (2, E) with:
      - bridge <-> element edges
      - parent_element -> element edges (and reverse)
    """
    edges = []

    b_ids = df[BRIDGE_ID_COL].map(normalize_bridge_id).values
    e_nums = df[ELEM_NUM_COL].astype("Int64").values
    p_nums = df[PARENT_ELEM_NUM_COL].astype("Int64").values

    for bid, en, pn in zip(b_ids, e_nums, p_nums):
        if bid is None or pd.isna(en):
            continue

        b_key = f"bridge:{bid}"
        e_key = f"elem:{bid}:{int(en)}"

        if b_key in node2idx and e_key in node2idx:
            b_i = node2idx[b_key]
            e_i = node2idx[e_key]
            # undirected / bidirectional
            edges.append((b_i, e_i))
            edges.append((e_i, b_i))

        # parent-child within same bridge if parent exists as element node
        if not pd.isna(pn):
            p_key = f"elem:{bid}:{int(pn)}"
            if p_key in node2idx and e_key in node2idx:
                p_i = node2idx[p_key]
                e_i = node2idx[e_key]
                edges.append((p_i, e_i))
                edges.append((e_i, p_i))

    # de-duplicate
    edges = list(set(edges))
    edge_index = np.array(edges, dtype=np.int64).T  # shape (2, E)
    return edge_index

def make_feature_frame(df: pd.DataFrame):
    """
    Creates a feature-ready dataframe:
      - converts booleans to 0/1
      - one-hot encodes remaining object/categorical columns (except IDs)
      - keeps numeric columns as-is
    IMPORTANT: we will remove leakage columns later (like TARGET, YEAR, IDs).
    """
    work = df.copy()

    # bool -> int
    bool_cols = [c for c in work.columns if work[c].dtype == bool]
    for c in bool_cols:
        work[c] = work[c].astype(np.int8)

    # Identify columns we do NOT want to one-hot
    id_like = {YEAR_COL, BRIDGE_ID_COL, "LATEST_STRUCTURE_NUMBER", "NBI_8 - Structure Number", "NBI_2025 NBI Structure Number"}
    # Keep element numbers numeric as-is (or you can drop them later)
    # We'll one-hot encode remaining object columns that aren’t IDs.
    obj_cols = [c for c in work.columns if work[c].dtype == "object" and c not in id_like]

    # One-hot encode object columns (can get large; but preserves info)
    work = pd.get_dummies(work, columns=obj_cols, dummy_na=True)

    return work

def build_yearly_matrices(df_raw: pd.DataFrame, node2idx: dict, elem_nodes: set, bridge_nodes: set):
    """
    Returns:
      X_list: list[np.ndarray] of shape (N, F)
      y_list: list[np.ndarray] of shape (N,) with NaN for non-element nodes or missing labels
      label_mask_list: list[np.ndarray] boolean mask for supervised nodes each year
      feature_cols: list[str]
    """
    df = df_raw.copy()
    df[BRIDGE_ID_COL] = df[BRIDGE_ID_COL].map(normalize_bridge_id)

    feat_df = make_feature_frame(df)

    # Drop leakage / identifiers from features
    drop_cols = [
        TARGET_COL, YEAR_COL, BRIDGE_ID_COL,
        "LATEST_STRUCTURE_NUMBER", "NBI_8 - Structure Number", "NBI_2025 NBI Structure Number"
    ]
    drop_cols = [c for c in drop_cols if c in feat_df.columns]
    feat_df_no_leak = feat_df.drop(columns=drop_cols)

    feature_cols = list(feat_df_no_leak.columns)
    N = len(node2idx)
    F = len(feature_cols)

    X_list, y_list, mask_list = [], [], []

    # Precompute node-type indicator (helps a lot in mixed node graphs)
    node_type = np.zeros((N, 2), dtype=np.float32)  # [is_bridge, is_element]
    for k, i in node2idx.items():
        if k.startswith("bridge:"):
            node_type[i, 0] = 1.0
        else:
            node_type[i, 1] = 1.0

    for year in YEARS:
        sub_raw = df[df[YEAR_COL] == year]
        sub_feat = feat_df_no_leak.loc[sub_raw.index]

        X = np.zeros((N, F + 2), dtype=np.float32)  # +2 for node type
        y = np.full((N,), np.nan, dtype=np.float32)

        # Fill element-node features row-by-row (this keeps everything)
        for ridx, row in sub_raw.iterrows():
            bid = row[BRIDGE_ID_COL]
            en = row[ELEM_NUM_COL]
            if pd.isna(bid) or pd.isna(en):
                continue

            e_key = f"elem:{bid}:{int(en)}"
            if e_key not in node2idx:
                continue
            ni = node2idx[e_key]

            X[ni, :F] = sub_feat.loc[ridx].to_numpy(dtype=np.float32, copy=False)
            # target label for this year (HI)
            if TARGET_COL in sub_raw.columns and not pd.isna(row[TARGET_COL]):
                y[ni] = float(row[TARGET_COL])

            # Bridge node features: you can optionally ALSO fill the bridge node by aggregating
            # one row per bridge-year. A simple way:
            # - pick first row of that bridge-year to populate the bridge node (since NBI fields repeat)
            # We'll do that after loop to avoid repeated assignments.

        # Populate bridge nodes using the first row per bridge in this year
        first_per_bridge = sub_raw.dropna(subset=[BRIDGE_ID_COL]).groupby(BRIDGE_ID_COL).head(1)
        for ridx, row in first_per_bridge.iterrows():
            bid = row[BRIDGE_ID_COL]
            b_key = f"bridge:{bid}"
            if b_key not in node2idx:
                continue
            bi = node2idx[b_key]
            X[bi, :F] = sub_feat.loc[ridx].to_numpy(dtype=np.float32, copy=False)

        # append node type indicator
        X[:, F:] = node_type

        # Mask: supervised nodes are element nodes with non-NaN y
        label_mask = ~np.isnan(y)

        X_list.append(X)
        y_list.append(y)
        mask_list.append(label_mask)

    return X_list, y_list, mask_list, feature_cols

c:\Users\aksel\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class TemporalGCN_GRU(nn.Module):
    def __init__(self, in_dim: int, gcn_hidden: int = 128, gru_hidden: int = 128):
        super().__init__()
        self.gcn1 = GCNConv(in_dim, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)

        self.gru = nn.GRU(input_size=gcn_hidden, hidden_size=gru_hidden, batch_first=False)
        self.head = nn.Linear(gru_hidden, 1)

    def forward(self, X_seq, edge_index):
        """
        X_seq: list[Tensor] length T, each (N, F)
        edge_index: Tensor (2, E)
        Returns:
          preds_next: list[Tensor] length T-1, each (N,) prediction for next step
        """
        embeddings = []
        for X in X_seq:
            h = F.relu(self.gcn1(X, edge_index))
            h = F.relu(self.gcn2(h, edge_index))
            embeddings.append(h)

        # Stack to (T, N, H) for GRU: time-first
        H = torch.stack(embeddings, dim=0)

        # GRU evolves node representations through time
        out, _ = self.gru(H)  # out: (T, N, gru_hidden)

        # Predict HI for each time step's "next year"
        # We'll interpret out[t] -> prediction of y at time t+1
        preds = self.head(out).squeeze(-1)  # (T, N)
        preds_next = preds[:-1]             # (T-1, N)
        return preds_next

In [ ]:
def train_one_step(model, X_list, y_list, mask_list, edge_index, lr=1e-3, epochs=200):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    edge_index_t = torch.tensor(edge_index, dtype=torch.long, device=device)
    X_seq = [torch.tensor(X, dtype=torch.float32, device=device) for X in X_list]
    y_seq = [torch.tensor(y, dtype=torch.float32, device=device) for y in y_list]
    m_seq = [torch.tensor(m, dtype=torch.bool, device=device) for m in mask_list]

    # We train on t=0..T-2 predicting y_{t+1}
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        preds_next = model(X_seq, edge_index_t)  # (T-1, N)

        losses = []
        for t in range(len(X_seq) - 1):
            y_true = y_seq[t + 1]
            mask = m_seq[t + 1]  # supervise on labels available at t+1

            # IMPORTANT: only element nodes have labels; bridge nodes remain masked out automatically
            loss = F.smooth_l1_loss(preds_next[t][mask], y_true[mask])
            losses.append(loss)

        loss_total = torch.stack(losses).mean()
        loss_total.backward()
        optimizer.step()

        if ep % 20 == 0:
            print(f"epoch {ep:04d} | loss {loss_total.item():.5f}")

    return model

In [ ]:
@torch.no_grad()
def rollout_5_years(model, X_list, edge_index, years_ahead=5):
    """
    Baseline rollout:
      - Use the last available feature snapshot (2024) repeated for future years
      - Predict HI forward recursively (note: HI is NOT part of features to avoid leakage)
    """
    device = next(model.parameters()).device
    edge_index_t = torch.tensor(edge_index, dtype=torch.long, device=device)

    X_seq = [torch.tensor(X, dtype=torch.float32, device=device) for X in X_list]
    last_X = X_seq[-1]

    preds_future = []
    # Start from the observed sequence (2020..2024)
    current_seq = X_seq[:]

    for k in range(years_ahead):
        preds_next = model(current_seq, edge_index_t)  # predicts for each step’s next; last one is what we need
        y_hat_next = preds_next[-1]                    # prediction for "next year" after last step
        preds_future.append(y_hat_next.detach().cpu().numpy())

        # For baseline covariates: append the same X again.
        # If you have forecasted covariates, you would build a new X_{t+1} here instead.
        current_seq = current_seq[1:] + [last_X]

    return preds_future  # list length 5, each (N,)